### Crear la tabla results_movie en la capa "gold"
##### Este notebook es una modificación del 10-results_movie original de la carpeta transformation, ese notebook estaba en lenguaje sql y solo creaba una tabla a partir de un select, pero esto no es muy eficiente, porque debemos cargar esa tabla a partir de los datos que se ingresen en la consulta select, además esa tabla se crea una sola vez y no estamos haciendo una validación para eliminar previamente esa carga.
##### Después, si hiciéramos esa validación de eliminar esa tabla y volver a cargar, habría un tema y es que los datos pueden crecer a partir de ese select, lo cual tampoco es muy eficiente.
##### Por esto ahora uso delta lake, creo una tabla vacía y guardo los datos del select en una vista temporal en vez de una tabla, finalmente hago un merge entre la tabla y la vista para crearla, esto es mucho más eficiente que el notebook original, en este caso haremos todo en lenguaje Python.

In [0]:
dbutils.widgets.text("p_file_date","2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
spark.sql("""
          CREATE TABLE IF NOT EXISTS movie_gold_inc.results_movie
          (
            year_release_date INT,
            country_name STRING,
            company_name STRING,
            budget FLOAT,
            revenue FLOAT,
            movie_id INT,
            country_id INT,
            company_id INT,
            created_date DATE,
            updated_date DATE
          )
          USING DELTA
          """)

In [0]:
spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW v_results_movie
        AS
        SELECT m.year_release_date, c.country_name, pco.company_name, m.budget, m.revenue, m.movie_id, c.country_id, pco.company_id
        FROM movie_silver_inc.movies m
        INNER JOIN movie_silver_inc.productions_countries pc ON m.movie_id = pc.movie_id
        INNER JOIN movie_silver_inc.countries c ON c.country_id = pc.country_id
        INNER JOIN movie_silver_inc.movies_companies mc ON m.movie_id = mc.movie_id
        INNER JOIN movie_silver_inc.production_companies pco ON mc.company_id = pco.company_id 
        WHERE m.file_date = '{v_file_date}'
 """)

In [0]:
%sql
SELECT * FROM v_results_movie

In [0]:
spark.sql("""
        MERGE INTO movie_gold_inc.results_movie tgt
        USING v_results_movie src
        ON (tgt.movie_id = src.movie_id and tgt.country_id = src.country_id and tgt.company_id = src.company_id)
        WHEN MATCHED THEN
        UPDATE SET
            year_release_date = src.year_release_date,
            country_name = src.country_name,
            company_name = src.company_name,
            budget = src.budget,
            revenue = src.revenue,
            updated_date = current_timestamp
        WHEN NOT MATCHED
        THEN INSERT (year_release_date, country_name, company_name, budget, revenue, movie_id, country_id, company_id, created_date)
        VALUES (src.year_release_date, src.country_name, src.company_name, src.budget, src.revenue, src.movie_id, src.country_id, src.company_id, current_timestamp)
""")


In [0]:
%sql
SELECT COUNT(*) FROM v_results_movie

In [0]:
%sql
SELECT COUNT(1) FROM movie_gold_inc.results_movie